# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an end-to-end example for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is fully referenced by `@id` fields for maximum reproducibility.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset schema/metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {getattr(metadata, 'name', '<no name>')}")
print(f"Description: {getattr(metadata, 'description', '<no description>')}")
print(f"Published: {getattr(metadata, 'datePublished', '<no date>')}")
print(f"Version: {getattr(metadata, 'version', '<no version>')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities are referenced by their `@id` fields.

In [ ]:
def get_record_sets(meta):
    """Return a list of record set objects from metadata."""
    rec_sets = getattr(meta, 'recordSet', [])
    if callable(rec_sets):
        rec_sets = rec_sets()
    return rec_sets

# List all record set `@id`s, field `@id`s, and column `@id`s
record_sets = get_record_sets(metadata)
print(f"Number of record sets: {len(record_sets)}")

for rs in record_sets:
    print(f"Record set @id: {getattr(rs, '@id', None)}  |  name: {getattr(rs, 'name', None)}")
    fields = getattr(rs, 'field', [])
    print("  Fields:")
    for f in fields:
        print(f"    Field @id: {getattr(f, '@id', None)} | name: {getattr(f, 'name', None)}")
        columns = getattr(f, 'column', [])
        if columns:
            print("      Columns:")
            for c in columns:
                print(f"        Column @id: {getattr(c, '@id', None)} | name: {getattr(c, 'name', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s as explored above.


In [ ]:
# Choose record set(s) for extraction. For demonstration, extract all.
record_set_ids = [getattr(rs, '@id') for rs in get_record_sets(metadata)]

dataframes = {}
for rec_id in record_set_ids:
    try:
        print(f"Extracting records for record set '@id': {rec_id}")
        recs = list(dataset.records(record_set=rec_id))
        if recs:
            df = pd.DataFrame(recs)
            dataframes[rec_id] = df
            print(f"Loaded {len(df)} rows. Columns: {df.columns.tolist()}")
        else:
            print(f"No records found for {rec_id}")
    except Exception as e:
        print(f"Error extracting {rec_id}: {e}")

# Display the columns of the first found record set
if dataframes:
    example_recset_id = list(dataframes.keys())[0]
    print(f"First DataFrame columns under '{example_recset_id}':")
    print(dataframes[example_recset_id].columns.tolist())
    dataframes[example_recset_id].head()
else:
    print("No dataframes extracted.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Select record set and field IDs to analyze
if dataframes:
    # Replace the IDs below with valid values found in previous cell
    rec_id = example_recset_id
    df = dataframes[rec_id]

    # Try to find a reasonable numeric field (e.g., protein abundances, MW, etc.)
    from numpy import number
    possible_numeric = [col for col in df.columns 
                       if pd.api.types.is_numeric_dtype(df[col]) and 'id' not in col.lower()]

    if possible_numeric:
        numeric_field = possible_numeric[0]
    else:
        print('No obvious numeric columns, using the first column.')
        numeric_field = df.columns[0]

    print(f"Selected numeric field: {numeric_field}")

    # Filter: Show only records where numeric_field > threshold (choose threshold)
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    filtered_df = df[df[numeric_field] > threshold]

    print(f"Filtered records with {numeric_field} > {threshold}")
    display(filtered_df.head())

    # Normalize the field
    if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
        filtered_df[f'{numeric_field}_normalized'] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

    # Group by an example field if present
    group_fields = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by '{group_field}':")
        display(grouped_df.head())
    else:
        print('No categorical fields to group by.')
else:
    print("No dataframes available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group (if group_field exists)
    if group_fields:
        plt.figure(figsize=(12, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No visualization possible - no dataframe loaded.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset was loaded directly from its Croissant JSON-LD schema.
- Record sets, fields, and columns were referenced by their `@id` for reproducibility.
- Example numeric attributes and group attributes were explored and visualized.
- Further analysis can be performed depending on experimental design or research focus.